# Relative 因子：按规则筛选 + 按相关方向等权融合

1. 用 **config** 中关于 **lag** 与 **名字** 的规则筛选显著 relative 因子
2. 按各因子与 y 的 **相关系数方向**（正/负）决定融合时的符号，做 **等权融合**

**依赖**：请先运行 **01、02、03**（或本 notebook 将自行重建 df_y、relative_factors 与回归结果）。

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
PCT_COL = config.PCT_COL

LAG_DAYS = getattr(config, 'REGRESSION_LAG_DAYS', [0, 1, 2, 3])
MAX_P = getattr(config, 'REGRESSION_MAX_PVALUE', 0.05)
FUSION_LAG = getattr(config, 'FUSION_LAG_ALLOWED', [0, 1, 2, 3])
FUSION_SHEET = getattr(config, 'FUSION_SHEET_INCLUDE', None)
FUSION_PAIR = getattr(config, 'FUSION_PAIR_INCLUDE', None)

## 1. 准备 df_y、relative_factors、回归结果 reg_df

若内存中无则重建（与 03 一致）。

In [2]:
def get_pair_cols(df):
    return [c for c in df.columns if c != DATE_COL]

try:
    df_y
except NameError:
    all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx') and not f.startswith(config.SKIP_FILE_PREFIX)]
    file_50 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI50)), None)
    file_1000 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI1000)), None)
    df50 = pd.read_excel(os.path.join(DATA_DIR, file_50))[[DATE_COL, PCT_COL]].dropna()
    df1000 = pd.read_excel(os.path.join(DATA_DIR, file_1000))[[DATE_COL, PCT_COL]].dropna()
    df50[DATE_COL] = pd.to_datetime(df50[DATE_COL]).dt.date
    df1000[DATE_COL] = pd.to_datetime(df1000[DATE_COL]).dt.date
    df50 = df50.rename(columns={PCT_COL: '涨跌幅_50'})
    df1000 = df1000.rename(columns={PCT_COL: '涨跌幅_1000'})
    df_y = df50.merge(df1000, on=DATE_COL, how='inner')
    df_y = df_y.dropna()
    df_y['y'] = df_y['涨跌幅_50'].astype(float) - df_y['涨跌幅_1000'].astype(float)
    df_y = df_y[[DATE_COL, 'y']].reset_index(drop=True)
    if getattr(config, 'CUTOFF_DATE', None):
        import datetime
        cut = config.CUTOFF_DATE
        if isinstance(cut, (tuple, list)) and len(cut) >= 3:
            cut = datetime.date(int(cut[0]), int(cut[1]), int(cut[2]))
        df_y = df_y[df_y[DATE_COL] <= cut]
try:
    relative_factors
    sheet_names
except NameError:
    factor_path = os.path.join(DATA_DIR, getattr(config, 'AUGMENTED_FACTOR_FILENAME', '因子_国家队_增强版_pair_zscore.xlsx'))
    sheets_raw = pd.read_excel(factor_path, sheet_name=None)
    sheet_names = [n for n in sheets_raw.keys() if n != 'Target']
    ROLLING_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 60)
    MIN_VALID = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 60)
    RELATIVE_PAIRS = getattr(config, 'RELATIVE_FACTOR_PAIRS', [])
    def normalize_date(df):
        d = df.copy()
        if 'Date' in d.columns and DATE_COL not in d.columns:
            d = d.rename(columns={'Date': DATE_COL})
        if DATE_COL in d.columns:
            d[DATE_COL] = pd.to_datetime(d[DATE_COL], errors='coerce').dt.date
        return d
    def rolling_zscore(series, window):
        x = pd.to_numeric(series, errors='coerce')
        valid = x.notna()
        x_for_roll = x.where(valid)
        roll_mean = x_for_roll.rolling(window=window, min_periods=MIN_VALID).mean()
        roll_std = x_for_roll.rolling(window=window, min_periods=MIN_VALID).std()
        z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
        return z.where(valid)
    augmented_sheets = {}
    for name in sheet_names:
        df = normalize_date(sheets_raw[name].copy())
        for col in get_pair_cols(df):
            if col in df.columns:
                df[col] = rolling_zscore(pd.to_numeric(df[col], errors='coerce'), ROLLING_WINDOW)
        augmented_sheets[name] = df
    relative_factors = {}
    for name in sheet_names:
        df_z = augmented_sheets[name]
        out = df_z[[DATE_COL]].copy()
        for code_a, code_b in RELATIVE_PAIRS:
            if code_a in df_z.columns and code_b in df_z.columns:
                out[f"{code_a}-{code_b}"] = (df_z[code_a] - df_z[code_b]).ffill()
        relative_factors[name] = out

df_y_sorted = df_y.sort_values(DATE_COL).reset_index(drop=True)
try:
    reg_df
except NameError:
    reg_results = []
    for name in sheet_names:
        df_f = relative_factors[name]
        for col in get_pair_cols(df_f):
            for lag in LAG_DAYS:
                df_yt = df_y_sorted[[DATE_COL]].copy()
                df_yt['y_target'] = df_y_sorted['y'].shift(-lag)
                m = df_yt.merge(df_f[[DATE_COL, col]], on=DATE_COL, how='inner').dropna()
                if len(m) < 10:
                    reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': np.nan, 'p_value': np.nan})
                    continue
                y, x = m['y_target'].astype(float), m[col].astype(float)
                r, p = pearsonr(y, x)
                reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': round(r, 4), 'p_value': p})
    reg_df = pd.DataFrame(reg_results)
    reg_df['abs_corr'] = reg_df['corr'].abs()

sig = reg_df[reg_df['p_value'] < MAX_P].copy()
print('显著因子数（p < {}）:'.format(MAX_P), len(sig))
print('reg_df 行数:', len(reg_df))

显著因子数（p < 0.05）: 10
reg_df 行数: 132


## 2. 按 config 规则筛选：lag 与名字

In [3]:
filtered = sig[sig['lag'].isin(FUSION_LAG)].copy()
if FUSION_SHEET is not None and len(FUSION_SHEET) > 0:
    filtered = filtered[filtered['sheet'].isin(FUSION_SHEET)]
if FUSION_PAIR is not None and len(FUSION_PAIR) > 0:
    mask = filtered['标的对'].apply(lambda s: any(p in s for p in FUSION_PAIR))
    filtered = filtered[mask]
filtered = filtered.reset_index(drop=True)

print('融合规则: FUSION_LAG_ALLOWED =', FUSION_LAG)
print('         FUSION_SHEET_INCLUDE =', FUSION_SHEET)
print('         FUSION_PAIR_INCLUDE =', FUSION_PAIR)
print('筛选后因子数:', len(filtered))
display(filtered.head)

融合规则: FUSION_LAG_ALLOWED = [1, 2, 3]
         FUSION_SHEET_INCLUDE = None
         FUSION_PAIR_INCLUDE = None
筛选后因子数: 10


<bound method NDFrame.head of                sheet                  标的对  lag     n    corr   p_value  \
0            pct_chg  510050.SH-510100.SH    1  1300 -0.1275  0.000004   
1            pct_chg  510050.SH-510100.SH    2  1299  0.0588  0.033970   
2            pct_chg  159915.SZ-159977.SZ    3  1301 -0.0578  0.036968   
3          netinflow  588080.SH-588050.SH    2  1031 -0.0664  0.032939   
4          netinflow  159915.SZ-159977.SZ    1  1304 -0.0768  0.005542   
5          netinflow  159915.SZ-159977.SZ    2  1303 -0.0706  0.010827   
6          netinflow  159915.SZ-159977.SZ    3  1302 -0.0661  0.017137   
7  NAV_iopv_discount  510050.SH-510100.SH    1  1301 -0.0806  0.003634   
8  NAV_iopv_discount  588080.SH-588050.SH    2  1031  0.0649  0.037260   
9        ETF申购赎回现金差额  510050.SH-510100.SH    1  1301 -0.0661  0.017101   

   abs_corr  
0    0.1275  
1    0.0588  
2    0.0578  
3    0.0664  
4    0.0768  
5    0.0706  
6    0.0661  
7    0.0806  
8    0.0649  
9    0.0661  >

In [4]:
filtered

,sheet,标的对,lag,n,corr,p_value,abs_corr
0,pct_chg,510050.SH-510100.SH,1,1300,-0.1275,0.000004,0.1275
1,pct_chg,510050.SH-510100.SH,2,1299,0.0588,0.033970,0.0588
2,pct_chg,159915.SZ-159977.SZ,3,1301,-0.0578,0.036968,0.0578
3,netinflow,588080.SH-588050.SH,2,1031,-0.0664,0.032939,0.0664
4,netinflow,159915.SZ-159977.SZ,1,1304,-0.0768,0.005542,0.0768
5,netinflow,159915.SZ-159977.SZ,2,1303,-0.0706,0.010827,0.0706
6,netinflow,159915.SZ-159977.SZ,3,1302,-0.0661,0.017137,0.0661
7,NAV_iopv_discount,510050.SH-510100.SH,1,1301,-0.0806,0.003634,0.0806
8,NAV_iopv_discount,588080.SH-588050.SH,2,1031,0.0649,0.037260,0.0649
9,ETF申购赎回现金差额,510050.SH-510100.SH,1,1301,-0.0661,0.017101,0.0661


## 3. 按相关方向等权融合（按 lag 对齐到同一预测日）

对每个 (sheet, 标的对) 取 |corr| 最大对应的 **lag** 与 **sign**。融合时按 **目标日 T** 对齐：因子 i 在 T 日的贡献 = sign_i × 因子 i 在 (T − lag_i) 日的值，使融合因子预测的是**同一天**的大盘减小盘。

In [5]:
# 只保留 corr > 0 的因子参与融合
filtered_pos = filtered[filtered['corr'] != 0].copy()
print('corr > 0 筛选: {} -> {} 条'.format(len(filtered), len(filtered_pos)))

# 每个 (sheet, 标的对) 取 corr 最大的一行，得到融合用 lag（全部 sign=+1）
best = filtered_pos.loc[filtered_pos.groupby(['sheet', '标的对'])['corr'].idxmax()][['sheet', '标的对', 'corr', 'lag']].copy()
best['sign'] = 1.0

# 交易日历：用 y 的日期作为目标日，按位置 +lag 对齐因子
all_dates = df_y_sorted[DATE_COL].drop_duplicates().tolist()
date_to_idx = {d: i for i, d in enumerate(all_dates)}
idx_to_date = {i: d for i, d in enumerate(all_dates)}
n_dates = len(all_dates)

merged = None
for _, row in best.iterrows():
    name, col, sign, lag = row['sheet'], row['标的对'], row['sign'], int(row['lag'])
    df_f = relative_factors[name][[DATE_COL, col]].copy()
    df_f['pos'] = df_f[DATE_COL].map(date_to_idx)
    df_f = df_f.dropna(subset=['pos'])
    df_f['target_pos'] = df_f['pos'].astype(int) + lag
    df_f = df_f[df_f['target_pos'].between(0, n_dates - 1)]
    df_f['target_date'] = df_f['target_pos'].map(idx_to_date)
    contrib = sign * pd.to_numeric(df_f[col], errors='coerce')
    out = df_f[['target_date']].copy()
    out[f'_x_{name}_{col}'] = contrib.values
    if merged is None:
        merged = out
    else:
        merged = merged.merge(out, on='target_date', how='outer')

date_col = DATE_COL
merged = merged.rename(columns={'target_date': date_col})
x_cols = [c for c in merged.columns if c != date_col]
if len(x_cols) == 0:
    raise ValueError('没有可融合的因子序列')

# 融合前标准化：每列除以自身标准差，消除尺度差异，使各因子权重真正相等
for c in x_cols:
    std_c = merged[c].std()
    if std_c > 1e-12:
        merged[c] = merged[c] / std_c

merged['fusion'] = merged[x_cols].mean(axis=1)
df_fusion = merged[[date_col, 'fusion']].dropna(subset=['fusion']).sort_values(date_col).reset_index(drop=True)

print('参与融合的因子（仅 corr > 0）:')
display(best)
print('融合序列长度:', len(df_fusion))
display(df_fusion.head(10))

# 输出融合因子时间序列到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_fusion = os.path.join(_out_dir, '04_fusion_timeseries.xlsx')
df_fusion.to_excel(_path_fusion, index=False)
print('已输出:', _path_fusion)
# 筛选出的、用于构成 fusion 的 relative 因子对（sheet, 标的对, corr, sign）
_path_const = os.path.join(_out_dir, '04_fusion_constituents.xlsx')
best.to_excel(_path_const, index=False)
print('已输出:', _path_const)

corr > 0 筛选: 10 -> 10 条
参与融合的因子（仅 corr > 0）:


,sheet,标的对,corr,lag,sign
9,ETF申购赎回现金差额,510050.SH-510100.SH,-0.0661,1,1.0
7,NAV_iopv_discount,510050.SH-510100.SH,-0.0806,1,1.0
8,NAV_iopv_discount,588080.SH-588050.SH,0.0649,2,1.0
6,netinflow,159915.SZ-159977.SZ,-0.0661,3,1.0
3,netinflow,588080.SH-588050.SH,-0.0664,2,1.0
2,pct_chg,159915.SZ-159977.SZ,-0.0578,3,1.0
1,pct_chg,510050.SH-510100.SH,0.0588,2,1.0


融合序列长度: 1302


,交易日期,fusion
0,2020-07-03,0.419227
1,2020-07-06,0.601274
2,2020-07-07,-0.394843
3,2020-07-08,-1.164886
4,2020-07-09,0.096218
5,2020-07-10,0.081599
6,2020-07-13,0.710961
7,2020-07-14,-0.989113
8,2020-07-15,-0.449387
9,2020-07-16,0.181880


已输出: /Users/amadeus/Desktop/p3_adjusted_program/factor_build/outputs/04_fusion_timeseries.xlsx
已输出: /Users/amadeus/Desktop/p3_adjusted_program/factor_build/outputs/04_fusion_constituents.xlsx


## 4. 汇总

- **filtered**：按 lag/名字规则筛选后的显著因子表  
- **best**：每个 (sheet, 标的对) 取 |corr| 最大对应的 sign  
- **df_fusion**：等权融合后的时序（交易日期, fusion）

In [6]:
print('df_fusion 日期范围:', df_fusion[DATE_COL].min(), '~', df_fusion[DATE_COL].max())
print('fusion 统计: mean = {:.4f}, std = {:.4f}'.format(df_fusion['fusion'].mean(), df_fusion['fusion'].std()))

df_fusion 日期范围: 2020-07-03 ~ 2025-11-13
fusion 统计: mean = 0.0095, std = 0.4031
